# **Part 2: Data Integration & Preparation**
This notebook combines the cleaned datasets from Part 1 to create analysis-ready DataFrames for rule-based customer segmentation and behavioral analysis. The final DataFrames are validated and exported for use in the next notebook.

## **Notebook Overview**
### **Part 1: Data Overview**
- Loading and inspecting all datasets (`orders`, `products`, `departments`, `orders products prior`, and `customers`).
- Checking key ID fields and row counts before combining datasets.

### **Part 2: Merging Data**
- **Joining DataFrames:** Merging the core DataFrames and verifying join accuracy by checking final row counts and `_merge` indicators.
- **Handling Missing Product Metadata:** Resolving missing product details post-merge while documenting data limitations for downstream analysis.
- **Exporting Datasets:** Saving the finalized DataFrames in `.pkl` format to preserve data types for the next notebook.

---

## **1. Data Overview**
This section reviews each cleaned dataset to confirm their structure and identify the variables available for integration. This provides a foundation for determining how the datasets should be merged to be used for analysis.

### Importing Libraries & Cleaned Datasets

In [1]:
import pandas as pd
import os

In [2]:
# Importing cleaned datasets. 
path = r'C:\Users\TanaT\Portfolio Projects\Instacart - Customer Segementation & Behavioral Analysis'
df_cust = pd.read_pickle(os.path.join(path, '02 Data', '02 Processed Data', 'customers_cleaned.pkl'))
df_orders = pd.read_pickle(os.path.join(path, '02 Data', '02 Processed Data', 'orders_cleaned.pkl'))
df_prods = pd.read_pickle(os.path.join(path, '02 Data', '02 Processed Data', 'products_cleaned.pkl'))
df_dept = pd.read_pickle(os.path.join(path, '02 Data', '02 Processed Data', 'reshaped_dept_clean.pkl'))
df_cart = pd.read_pickle(os.path.join(path, '02 Data', '02 Processed Data', 'cart_behavior_cleaned.pkl')) 

### Customers Data

In [3]:
df_cust.head()

,user_id,gender,state,age,date_joined,n_dependents,fam_status,income
0,26711,Female,Missouri,48,2017-01-01,3,married,165665
1,33890,Female,New Mexico,36,2017-01-01,0,single,59285
2,65803,Male,Idaho,35,2017-01-01,2,married,99568
3,125935,Female,Iowa,40,2017-01-01,0,single,42049
4,130797,Female,Maryland,26,2017-01-01,1,married,40374


In [4]:
df_cust.shape

(206209, 8)

### Orders Data

In [5]:
df_orders.head()

,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order,is_first_order
0,2539329,1,1,Monday,8,NaN,True
1,2398795,1,2,Tuesday,7,15.0,False
2,473747,1,3,Tuesday,12,21.0,False
3,2254736,1,4,Wednesday,7,29.0,False
4,431534,1,5,Wednesday,15,28.0,False


In [6]:
df_orders['order_id'].nunique()

3421083

In [7]:
df_orders.shape

(3421083, 7)

In [8]:
df_orders.isnull().sum()

order_id                       0
user_id                        0
order_number                   0
order_dow                      0
order_hour_of_day              0
days_since_prior_order    206209
is_first_order                 0
dtype: int64

### Products Data

In [9]:
df_prods.head()

,product_id,product_name,aisle_id,department_id,price
0,1,Chocolate Sandwich Cookies,61,19,5.8
1,2,All-Seasons Salt,104,13,9.3
2,3,Robust Golden Unsweetened Oolong Tea,94,7,4.5
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1,10.5
4,5,Green Chile Anytime Sauce,5,13,4.3


In [10]:
df_prods.shape

(49668, 5)

### Departments Data

In [11]:
df_dept.head()

,department_id,department
0,1,frozen
1,2,other
2,3,bakery
3,4,produce
4,5,alcohol


### Orders Products Prior Data

In [12]:
df_cart.head()

,order_id,product_id,add_to_cart_order,reordered
0,2,33120,1,True
1,2,28985,2,True
2,2,9327,3,False
3,2,45918,4,True
4,2,30035,5,False


In [13]:
df_cart['order_id'].nunique()

3214874

In [14]:
df_cart.shape

(32434489, 4)

---

## **2. Merging Datasets**
The perviously cleaned DataFrames are merged to combine related variables while preserving the appropriate level of analysis.

### **Product + Department Integration**
- Product and department information were combined to associate each product with its corresponding department name, creating a more complete product-level dataset for subsequent analysis.
- The datasets were merged using the shared `department_id` key.
- A **left merge** was used to retain all products while matching each product to its corresponding department information.

In [15]:
# Combining dept and products
df_prods_dept = df_prods.merge(df_dept, on='department_id', how='left', indicator=True)

In [16]:
df_prods_dept.shape

(49668, 7)

In [17]:
df_prods_dept.head()

,product_id,product_name,aisle_id,department_id,price,department,_merge
0,1,Chocolate Sandwich Cookies,61,19,5.8,snacks,both
1,2,All-Seasons Salt,104,13,9.3,pantry,both
2,3,Robust Golden Unsweetened Oolong Tea,94,7,4.5,beverages,both
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1,10.5,frozen,both
4,5,Green Chile Anytime Sauce,5,13,4.3,pantry,both


In [18]:
df_prods_dept.isnull().sum()

product_id       0
product_name     0
aisle_id         0
department_id    0
price            0
department       0
_merge           0
dtype: int64

In [19]:
df_prods_dept['_merge'].value_counts()

_merge
both          49668
left_only         0
right_only        0
Name: count, dtype: int64

#### **Merge Validation Check**
No missing values were introduced by the merge, confirming that each product was successfully matched to a department name.

---

### **Order + Product Integration**
- The `df_cart` dataset contains product-level records for customers’ prior orders, while `df_orders` contains order-level information such as order sequence, timing, and days since the previous order.
- The datasets were merged on `order_id` to combine order-level behavioral information with the corresponding product-level purchase records.
- A **left merge** was used to retain all product-level records from `df_cart` while adding the corresponding order-level information.

In [20]:
df_orders_cart = df_cart.merge(df_orders, on='order_id', how='left', indicator=True)

In [21]:
df_orders_cart.shape

(32434489, 11)

In [22]:
# Checking number of unique orders in merged data. 
df_orders_cart['order_id'].nunique()

3214874

In [23]:
df_orders_cart.head()

,order_id,product_id,add_to_cart_order,reordered,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order,is_first_order,_merge
0,2,33120,1,True,202279,3,Thursday,9,8.0,False,both
1,2,28985,2,True,202279,3,Thursday,9,8.0,False,both
2,2,9327,3,False,202279,3,Thursday,9,8.0,False,both
3,2,45918,4,True,202279,3,Thursday,9,8.0,False,both
4,2,30035,5,False,202279,3,Thursday,9,8.0,False,both


In [24]:
df_orders_cart.isnull().sum()

order_id                        0
product_id                      0
add_to_cart_order               0
reordered                       0
user_id                         0
order_number                    0
order_dow                       0
order_hour_of_day               0
days_since_prior_order    2078068
is_first_order                  0
_merge                          0
dtype: int64

In [25]:
df_orders_cart['_merge'].value_counts()

_merge
both          32434489
left_only            0
right_only           0
Name: count, dtype: int64

In [26]:
# Checking that orders with nulls matches the original number. 
null_orders_count = df_orders_cart[df_orders_cart['days_since_prior_order'].isnull()]['order_id'].nunique()

print(f"Unique orders with nulls: {null_orders_count}")

Unique orders with nulls: 206209


#### **Merge Validation Check**
- The merge indicator confirmed that all product-level records in `df_cart` were successfully matched to an order in `df_orders`.
- No new missing values were introduced by the merge. The existing missing values in `days_since_prior_order` were retained from `df_orders` and correspond to customers’ first orders, as established during the initial data cleaning process.
- The number of unique orders with null `days_since_prior_order` values was 206,209, matching the total number of unique user IDs. This supports the assumption that each customer has one first order for which no previous-order interval exists.

---

### **Order + Product + Department Integration**
- The `df_orders_cart` dataset was merged with the `df_prods_dept` dataset to add product attributes, including `product_name`, `department` information, and `price` to the historical order records.
- The datasets were merged on `product_id`.
- A **left merge** was used to retain all historical product-order records in `df_orders_cart` while adding the corresponding product and department information.

In [27]:
# Dropping merge indicator columns.
df_prods_dept.drop(columns='_merge', inplace=True)
df_orders_cart.drop(columns='_merge', inplace=True)

In [28]:
df_order_behavior = df_orders_cart.merge(df_prods_dept, on='product_id', how='left', indicator='merge_status')

In [29]:
df_order_behavior.shape

(32434489, 16)

In [30]:
# Checking all orders survived the merge. 
df_order_behavior['order_id'].nunique()

3214874

In [31]:
df_order_behavior.head()

,order_id,product_id,add_to_cart_order,reordered,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order,is_first_order,product_name,aisle_id,department_id,price,department,merge_status
0,2,33120,1,True,202279,3,Thursday,9,8.0,False,Organic Egg Whites,86.0,16.0,11.3,dairy eggs,both
1,2,28985,2,True,202279,3,Thursday,9,8.0,False,Michigan Organic Kale,83.0,4.0,13.4,produce,both
2,2,9327,3,False,202279,3,Thursday,9,8.0,False,Garlic Powder,104.0,13.0,3.6,pantry,both
3,2,45918,4,True,202279,3,Thursday,9,8.0,False,Coconut Butter,19.0,13.0,8.4,pantry,both
4,2,30035,5,False,202279,3,Thursday,9,8.0,False,Natural Sweetener,17.0,13.0,13.7,pantry,both


In [32]:
df_order_behavior['merge_status'].value_counts()

merge_status
both          32399162
left_only        35327
right_only           0
Name: count, dtype: int64

In [33]:
# Checking if there are nulls after merge. Expected to be missing product data. 
df_order_behavior.isnull().sum()

order_id                        0
product_id                      0
add_to_cart_order               0
reordered                       0
user_id                         0
order_number                    0
order_dow                       0
order_hour_of_day               0
days_since_prior_order    2078068
is_first_order                  0
product_name                35327
aisle_id                    35327
department_id               35327
price                       35327
department                  35327
merge_status                    0
dtype: int64

In [34]:
# Percentage of missing data per column.
df_order_behavior.isnull().sum() * 100 / len(df_order_behavior)

order_id                  0.000000
product_id                0.000000
add_to_cart_order         0.000000
reordered                 0.000000
user_id                   0.000000
order_number              0.000000
order_dow                 0.000000
order_hour_of_day         0.000000
days_since_prior_order    6.406970
is_first_order            0.000000
product_name              0.108918
aisle_id                  0.108918
department_id             0.108918
price                     0.108918
department                0.108918
merge_status              0.000000
dtype: float64

#### **Merge Validation Check**
- The merge indicator showed that **32,399,162 order-product records** were successfully matched to product and department information.
- **35,327 records** were classified as `left_only`, indicating that these order-product records had no corresponding match in the product/department dataset.
- No `right_only` records were identified, confirming that all product/department records were associated with at least one order-product record.
- The unmatched records resulted in missing values for `product_name`, `aisle_id`, `department_id`, `department`, and `price`, affecting approximately **0.11% of order-product records**.

---

### **Investigating Missing Product Data**
The missing product information was reviewed to determine the cause of the unmatched records and whether they should be retained or excluded from the analysis.


In [35]:
# Finding only the rows where product name is missing.
missing_products = df_order_behavior[df_order_behavior['product_name'].isnull()]

In [36]:
missing_products.head()

,order_id,product_id,add_to_cart_order,reordered,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order,is_first_order,product_name,aisle_id,department_id,price,department,merge_status
347,43,21553,6,True,39630,14,Thursday,17,4.0,False,NaN,NaN,NaN,NaN,NaN,left_only
357,44,2240,9,False,183833,3,Friday,17,1.0,False,NaN,NaN,NaN,NaN,NaN,left_only
1985,224,34,8,True,109534,6,Saturday,14,13.0,False,NaN,NaN,NaN,NaN,NaN,left_only
4842,508,34,2,True,51003,6,Thursday,11,10.0,False,NaN,NaN,NaN,NaN,NaN,left_only
5054,537,1511,1,True,180135,15,Monday,8,3.0,False,NaN,NaN,NaN,NaN,NaN,left_only


In [37]:
missing_products.shape

(35327, 16)

#### **Investigation Findings**
Product-level information is missing across `product_name`, `aisle_id`, `department`, `department_id`, and `price` columns for the unmatched records.

#### Checking Number of Missing Products

In [38]:
# Finding how many products are missing by finding the number of unique product ids.  
missing_ids = missing_products['product_id'].nunique()
missing_ids

20

In [39]:
# Missing product ids. 
missing_ids = missing_products['product_id'].unique()
missing_ids

array([21553,  2240,    34,  1511,  4790,   116,  6799, 33664,   262,
        3230,  1780, 40440,  2586,  4283,    69,  3159,  3736,   525,
       26519, 38183])

#### Are missing products more common in first orders?

In [40]:
# Calculating the first-order rate among observations involving missing products.
missing_in_first = df_order_behavior[df_order_behavior['product_id'].isin(missing_ids)]['is_first_order'].mean()
print(f"Percentage of missing products in first orders: {missing_in_first:.2%}")

Percentage of missing products in first orders: 6.67%


In [41]:
# Baseline of first order rate.
df_order_behavior['is_first_order'].mean()

np.float64(0.06406970062022559)

#### How many first orders contained a missing product?

In [42]:
# How many unique first orders had at least one missing product?
affected_first_orders = df_order_behavior[
    (df_order_behavior['days_since_prior_order'].isnull()) & # first order and 
    (df_order_behavior['product_name'].isnull())             # missing product name 
]['order_id'].nunique()

print(f"Number of first orders affected: {affected_first_orders}")

Number of first orders affected: 2344


In [43]:
# Proportion of first orders affected. 
2344/206209*100

1.1367108128161234

#### **Investigation Findings**
- Found **20 unique products** are missing from the merged dataset, representing approximately **0.04% of the 49,668 unique products** in the cleaned product dataset.
- Approximately **6.67% of observations involving these missing products occur on first orders**, compared with a **6.41% overall first-order rate**. The similar rates suggest that the missing products are not disproportionately associated with first-order behavior, although this comparison alone does not establish that missingness is random.

---

### **Integrating Customer Demographics**
- Customer demographic information was added to the order-level product dataset to incorporate customer attributes such as `age`, `income`, number of dependents, and family status.
- The datasets were merged on `user_id`.
- A **left merge** was used to retain all order-product records while adding the corresponding customer information.

In [44]:
# Finding name of indicator column to drop.
df_order_behavior.columns

Index(['order_id', 'product_id', 'add_to_cart_order', 'reordered', 'user_id',
       'order_number', 'order_dow', 'order_hour_of_day',
       'days_since_prior_order', 'is_first_order', 'product_name', 'aisle_id',
       'department_id', 'price', 'department', 'merge_status'],
      dtype='object')

In [45]:
# Dropping the merge indicator column.
df_order_behavior.drop(columns=['merge_status'], inplace=True)

In [46]:
# Checking column was removed. 
df_order_behavior.columns

Index(['order_id', 'product_id', 'add_to_cart_order', 'reordered', 'user_id',
       'order_number', 'order_dow', 'order_hour_of_day',
       'days_since_prior_order', 'is_first_order', 'product_name', 'aisle_id',
       'department_id', 'price', 'department'],
      dtype='object')

In [47]:
# Adding customer data to merged data.
df_cust_segmentation = df_order_behavior.merge(df_cust, on='user_id', how='left', indicator='merge_check')

In [48]:
df_cust_segmentation.shape

(32434489, 23)

In [49]:
# Checking all orders survived the merge.
df_cust_segmentation['order_id'].nunique()

3214874

In [50]:
df_cust_segmentation.head()

,order_id,product_id,add_to_cart_order,reordered,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order,is_first_order,...,price,department,gender,state,age,date_joined,n_dependents,fam_status,income,merge_check
0,2,33120,1,True,202279,3,Thursday,9,8.0,False,...,11.3,dairy eggs,Male,Idaho,57,2020-02-06,3,married,98119,both
1,2,28985,2,True,202279,3,Thursday,9,8.0,False,...,13.4,produce,Male,Idaho,57,2020-02-06,3,married,98119,both
2,2,9327,3,False,202279,3,Thursday,9,8.0,False,...,3.6,pantry,Male,Idaho,57,2020-02-06,3,married,98119,both
3,2,45918,4,True,202279,3,Thursday,9,8.0,False,...,8.4,pantry,Male,Idaho,57,2020-02-06,3,married,98119,both
4,2,30035,5,False,202279,3,Thursday,9,8.0,False,...,13.7,pantry,Male,Idaho,57,2020-02-06,3,married,98119,both


In [51]:
df_cust_segmentation['merge_check'].value_counts()

merge_check
both          32434489
left_only            0
right_only           0
Name: count, dtype: int64

In [52]:
df_cust_segmentation.isnull().sum()

order_id                        0
product_id                      0
add_to_cart_order               0
reordered                       0
user_id                         0
order_number                    0
order_dow                       0
order_hour_of_day               0
days_since_prior_order    2078068
is_first_order                  0
product_name                35327
aisle_id                    35327
department_id               35327
price                       35327
department                  35327
gender                          0
state                           0
age                             0
date_joined                     0
n_dependents                    0
fam_status                      0
income                          0
merge_check                     0
dtype: int64

#### **Merge Validation Check**
- The merge indicator confirmed that all order-product records were successfully matched to a `user_id`.
- No new missing customer attributes were introduced by the merge. Existing missing values in `days_since_prior_order` and product metadata were retained from the previous merge.

---

### **Handling Missing Product Metadata**
- Product-order records with missing product metadata (`product_name`, `department`, `department_id`, `aisle_id`, and `price`) were removed because these attributes are required for product-level behavioral analysis.

In [53]:
# Removing rows with missing product data specifically.
df_merged_clean = df_cust_segmentation.dropna(subset=['product_name'])

In [54]:
df_merged_clean.shape

(32399162, 23)

In [55]:
# Checking orders. 
df_merged_clean['order_id'].nunique()

3214668

In [56]:
df_merged_clean.isnull().sum()

order_id                        0
product_id                      0
add_to_cart_order               0
reordered                       0
user_id                         0
order_number                    0
order_dow                       0
order_hour_of_day               0
days_since_prior_order    2075711
is_first_order                  0
product_name                    0
aisle_id                        0
department_id                   0
price                           0
department                      0
gender                          0
state                           0
age                             0
date_joined                     0
n_dependents                    0
fam_status                      0
income                          0
merge_check                     0
dtype: int64

#### Checking Number of Orders Affected Due to Dropping Nulls

In [57]:
lost_orders = set(df_order_behavior['order_id']) - set(df_merged_clean['order_id'])
len(lost_orders)

206

In [58]:
df_orders['order_id'].nunique() 

3421083

In [59]:
# Proportion of dropped orders. 
206 / 3421083 * 100

0.006021485009279225

In [60]:
# Checking if removing the product nulls affected users. 
df_merged_clean['user_id'].nunique()

206209

### **Cleaning Impact**
- The missing product metadata affected **35,327 order-product rows**, representing approximately **0.1% of all order-product records**.
- Removing these rows resulted in the loss of **206 unique orders (~0.006% of all orders)** while retaining the full customer base of **206,209 users**.
- The resulting dataset contains only product-order records with the product attributes needed for the customer segmentation analysis.

***Limitation:*** Removing these records may slightly underestimate basket size for orders that contained both complete and incomplete product records, as products with missing metadata are excluded from the final dataset.

---

## **Exporting Merged and Cleaned Datasets**
The final DataFrames are exported as pickle files (`.pkl`) to preserve their pandas data structures, data types, and associated metadata. The exported files are ready for use in the customer segmentation and behavioral analysis performed in the next notebook.

In [61]:
# Products table (products and departments - all products, no nulls)
df_prods_dept.to_pickle(os.path.join(path, '02 Data', '02 Processed Data', 'products_table.pkl'))

# Order-product table (product ID, only nulls for first orders - cart behavior)
df_orders_cart.to_pickle(os.path.join(path, '02 Data', '02 Processed Data', 'order_products_prior.pkl'))

# Order-product table with product metadata (contains nulls)
df_order_behavior.to_pickle(os.path.join(path, '02 Data', '02 Processed Data', 'order_product_names_prior.pkl'))

# Original merge with product nulls.
df_cust_segmentation.to_pickle(os.path.join(path, '02 Data', '02 Processed Data', 'merged_data_original.pkl'))

# Clean merge no nulls (for analysis).
df_merged_clean.to_pickle(os.path.join(path, '02 Data', '02 Processed Data', 'merged_data_clean.pkl'))